Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from random import randint

inputs_4regimes = np.genfromtxt("db_4regimes_input.csv", delimiter=",", dtype=np.float32)[:, :-1]
outputs_4regimes = np.genfromtxt("db_4regimes_output.csv", delimiter=",", dtype=np.float32)[:, :-1]
inputs_z = np.genfromtxt("db_regime_z_input.csv", delimiter=",", dtype=np.float32)[:, :-1]
outputs_z = np.genfromtxt("db_regime_z_output.csv", delimiter=",", dtype=np.float32)[:, :-1]
inputs_helico = np.genfromtxt("db_regime6_helico_input.csv", delimiter=",", dtype=np.float32)[:, :-1]
outputs_helico = np.genfromtxt("db_regime6_helico_output.csv", delimiter=",", dtype=np.float32)[:, :-1]


Stacking datasets

In [ ]:
inputs = np.concatenate([inputs_4regimes, inputs_z, inputs_helico], axis=0).reshape(6250, -1, 4)
outputs = np.concatenate([outputs_4regimes, outputs_z, outputs_helico], axis=0).reshape(6250, -1, 18)

outputs.shape

In [ ]:
data_names = "x y z vx vy vz ax ay az phi tetha psi wx wy wz wpointx wpointy wpointz".split(' ')

In [ ]:
for i in range(18):
    print(f"Moyenne {data_names[i]} : {np.max(outputs[:, :, i])}")

Converting to thrust yaw etc

In [ ]:
def drone_commands(input_array):
    M = np.array([[1,  1,  1,  1],
                  [1, -1,  1, -1],
                  [1, -1, -1,  1],
                  [1,  1, -1, -1]])
    return input_array @ M.T
commands = drone_commands(inputs)

Visualizing data samples

In [ ]:
fig, axes = plt.subplots(len(data_names) // 3, 3)
fig.set_figheight(15)
fig.set_figwidth(15)
rand_idx = randint(0, 6250)

for i in range(len(data_names)):
    axes[i//3, i%3].plot(normalized_outputs[rand_idx, :, i])

fig2, axes2 = plt.subplots(1, 4)
fig2.set_figheight(6)
fig2.set_figwidth(12)
for i in range(4):
    axes2[i].plot(commands[rand_idx, :, i])

plt.show()

Deleting outliers

In [ ]:
def outlier_bound(signal):
    height_limit = 30
    distance_limit = 70
    angle_limit = 70
    
    is_outlier = False
    is_outlier = is_outlier or np.max(np.abs(signal[:, 0])) > distance_limit
    is_outlier = is_outlier or np.max(np.abs(signal[:, 1])) > distance_limit
    is_outlier = is_outlier or np.max(np.abs(signal[:, 2])) > height_limit
    
    is_outlier = is_outlier or np.max(np.abs(signal[:, 9])) > angle_limit
    is_outlier = is_outlier or np.max(np.abs(signal[:, 10])) > angle_limit
    
    return is_outlier

Saving the dataset for chosen outputs

In [ ]:
def save_dataset(inputs4, outputs18, 
                 save_idxs=[6, 7, 8, 15, 16, 17], 
                 outlier_function=None, 
                 inputs_name="db_inputs0.csv", outputs_name="db_outputs0.csv",
                 convert_to_commands=True):
    p_saving = np.random.permutation(inputs4.shape[0])
    inputs_permuted = inputs4[p_saving].reshape(inputs4.shape[0], -1, 4)
    outputs_permuted = outputs18[p_saving].reshape(inputs4.shape[0], -1, 18)
    
    if outlier_function is not None:
        without_outlier = np.array([signal for signal in outputs_permuted if not outlier_function(signal)])
        print(f"without outliers : {without_outlier.shape[0]}, initial total : {outputs.shape[0]}")
        input_without_outlier = np.array([val for (val, signal) in zip(inputs_permuted, outputs_permuted) if not outlier(signal)])
    else:
        without_outlier = outputs_permuted
        input_without_outlier = inputs_permuted
        
    if convert_to_commands:
        input_commands = drone_commands(input_without_outlier)
    else:
        input_commands = input_without_outlier
        
    inputs_commands = inputs_commands.reshape(without_outlier.shape[0], -1)
    outputs_to_save = without_outlier[:, :, np.isin(np.arange(18) % 18, save_idxs)].reshape(without_outlier.shape[0], -1)
    
    np.savetxt(inputs_name, inputs_commands, delimiter=",")
    np.savetxt(outputs_name, outputs_to_save, delimiter=",")
    

In [ ]:
data_idxs = {
    "pos":[0, 1, 2, 9, 10, 11],
    "vel":[3, 4, 5, 12, 13, 14],
    "acc":[6, 7, 8, 15, 16, 17]
}